# ⚛️ Pulser-Gym: Interactive Demo

Welcome to **Pulser-Gym**. This interactive notebook bridges standard classical Deep Reinforcement Learning with Neutral Atom Quantum Computing!

We're wrapping Pasqal's `Pulser` emulator inside an OpenAI `Gymnasium` interface. This allows us to use off-the-shelf RL agents to figure out exactly how to tune the physical amplitude of a quantum laser. No complex quantum mechanics required natively—just straightforward mathematical optimization targeting the NP-Hard Maximum Independent Set (MIS) graph problem!

Let's dive in.

## 1. Installation and Library Injection
First, let's load all the necessary components of our engine. Feel free to execute `!pip install -r pulser-gym-env/requirements.txt` if you are running this natively in Colab or a fresh workspace.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Connect to our custom physical bridge package locally
sys.path.append(os.path.abspath('pulser-gym-env'))

from pulser import Register
from stable_baselines3 import PPO
from pulser_gym.env_01_core import PulserEnv
from pulser_gym.sequence_02_translation import build_pulse_sequence

print("Hardware and Agent modules loaded successfully.")

## 2. Visualizing the Quantum Hardware (The "Game Board")
To solve the Maximum Independent Set problem computationally, we load atoms into a physical register. Here, we lay out 9 Rubidium atoms in a strict 2D Square matrix grid (3x3 grid, spaced 5 $\mu$m apart). 

Our Reinforcement Learning agent wants to find a solution that allows multiple atoms to be excited simultaneously... but if two *physically adjacent* atoms become excited, the physical Rydberg blockade (Euclidean radius) prevents it, mathematically resulting in heavy penalties!

In [ ]:
n_atoms = 9
side_length = int(np.sqrt(n_atoms))
graph_register = Register.square(side_length, spacing=5.0)

fig, ax = plt.subplots(figsize=(6, 4))
graph_register.draw(with_labels=True, custom_ax=ax)
ax.set_title(f"2D Square Lattice Topology: {side_length}x{side_length} Neutral Atoms")
plt.show()

## 3. Inspecting the Gymnasium API
Let's connect the Qutip emulation backend directly to standard RL bounds. Notice that the Continuous Action space has 1 dimension (Laser Amplitude scaling), and the Observation Space has 9 dimensions (the statistical modal likelihood probability vector mapping over each atom in our 3x3 array!).

In [ ]:
env = PulserEnv(n_qubits=n_atoms)

print(f"Agent Legal Action Bounds (Scaling 0 to 1): {env.action_space}")
print(f"Environment Mathematical State Boundary: {env.observation_space}")

## 4. The Agent: Proximal Policy Optimization (PPO)
We will now boot an off-the-shelf multi-layer perceptron running Stable-Baselines3. We allow the model to interact with the environment for 1,024 timesteps. 

The PPO algorithm learns to adjust its floating-point output to maximize the positive MIS reward while avoiding negative graph collision physics natively across the grid.

In [ ]:
print("Bootstrapping RL Optimization Loop...")
model = PPO("MlpPolicy", env, verbose=0) # Set verbose=1 to see training STDOUT in standard modes
model.learn(total_timesteps=1024)

print("\nTraining Complete! Policy mapped.")

## 5. Final Inference: The Quantum Sequence and Graph Validation
We've optimized the model. Let's let the agent interact securely with a deterministic graph state to extract its final "learned" optimal physical physics waveform (e.g., Amplitude targeting).

We'll plot the actual generated physics Pulse across time!

In [ ]:
# Reset Env to start from scratch
obs, _ = env.reset()

# Get deterministic prediction from the agent
action, _ = model.predict(obs, deterministic=True)

# Generate and extract the real waveform
generated_sequence = build_pulse_sequence(n_atoms, action)
amplitude = action[0] * 10.0 # Restoring scale

# Let the environment compute the physics
final_obs, final_reward, _, _, _ = env.step(action)

print("--- AI Quantum Physics Result ---")
print(f"Optimal Laser Amplitude Issued: {amplitude:.3f} rad/us")
print(f"Terminal Quantum String Produced: {final_obs}")
print(f"Final Graph MIS Evaluation Score: {final_reward:.1f}")

# Plotting the raw physical target
fig_seq = generated_sequence.draw(draw_halfbank=False, show=False)